# RRG (k=4): Simulation vs AME with Resetting

This notebook computes ensemble-averaged time evolution on a random regular graph (RRG) with degree $k=4$ and compares:
- Monte Carlo simulation
- integrated AME differential equations

Current setup uses **resetting** ($r>0$).

To avoid clutter, the comparison focuses on **two selected** $s_{4,m}(t)$ components.

In [136]:
using Graphs
using Random
using Statistics
using Printf
using Plots

project_root = isdir(joinpath(pwd(), "src")) ? pwd() : dirname(dirname(pwd()))

# Always reload local module so new API changes are available in this kernel.
include(joinpath(project_root, "src", "VoterResetting.jl"))

Main.VoterResetting

In [137]:
# Simulation parameters (with resetting)
N = 100000
k = 4
m0 = 0.2
r = 3

nsamples = 5000
times = collect(range(0.0, 10, length=101))
seed = 1234

# Keep only two s_{k,m} components for clearer plots
selected_s_m = [0, 2, 4]

# Use the SAME reset protocol in simulation and AME
delta_reset_protocol = VoterResetting.delta_reset(m0)

Random.seed!(seed)
G = random_regular_graph(N, k)
params = VoterResetting.ComplexParams(r, m0)

println("RRG ready (RESETTING): N=$N, k=$k, m0=$m0, r=$r, nsamples=$nsamples, ntimes=$(length(times)), selected_s_m=$(selected_s_m), reset=delta_reset($m0)")
println("Convention: simulation and AME are compared on the same t-grid.")

RRG ready (RESETTING): N=100000, k=4, m0=0.2, r=3, nsamples=5000, ntimes=101, selected_s_m=[0, 2, 4], reset=delta_reset(0.2)
Convention: simulation and AME are compared on the same t-grid.


In [ ]:
result = VoterResetting.simulate_degree_evolution_complex(
    G,
    params;
    reset=delta_reset_protocol,
    times=times,
    nsamples=nsamples,
    mode=:all,
)

@assert k in result.k_values "Requested k is not present in graph degree classes"
k_idx = findfirst(==(k), result.k_values)

s4 = result.s_values[k_idx]  # size: (ntimes, k+1), columns m=0..k
i4 = result.i_values[k_idx]  # size: (ntimes, k+1), columns m=0..k

sum_s = vec(sum(s4, dims=2))
sum_i = vec(sum(i4, dims=2))
sum_si = sum_s + sum_i
m_from_si = vec(sum(i4 .- s4, dims=2))

@printf("Max |sum(s+i)-1| over time: %.3e\n", maximum(abs.(sum_si .- 1.0)))

In [ ]:
# Plot all s_{4,m}(t)
p_s = plot(size=(900, 420), legend=:right, xlabel="t", ylabel="fraction", title="RRG k=4: s_{4,m}(t)")
for m in 0:k
    plot!(p_s, result.times, s4[:, m + 1], lw=2, label="s_{$k,$m}")
end
display(p_s)

# Plot all i_{4,m}(t)
p_i = plot(size=(900, 420), legend=:right, xlabel="t", ylabel="fraction", title="RRG k=4: i_{4,m}(t)")
for m in 0:k
    plot!(p_i, result.times, i4[:, m + 1], lw=2, ls=:dash, label="i_{$k,$m}")
end
display(p_i)

# Aggregate checks and reconstructed magnetization
p_agg = plot(size=(900, 420), legend=:right, xlabel="t", ylabel="value", title="RRG k=4: aggregates and m(t)")
plot!(p_agg, result.times, sum_s, lw=2, label="sum_m s_{$k,m}")
plot!(p_agg, result.times, sum_i, lw=2, label="sum_m i_{$k,m}")
plot!(p_agg, result.times, sum_si, lw=2, ls=:dot, label="sum_m (s+i)")
plot!(p_agg, result.times, m_from_si, lw=2, ls=:dash, label="m(t) = sum_m(i-s)")
hline!(p_agg, [m0], ls=:dashdot, lw=1.5, label="m0")
display(p_agg)

In [ ]:
# Compare Monte Carlo simulation vs integrated AME equations on the same RRG case.
ame = VoterResetting.solve_ame_evolution(
    G,
    m0,
    r,
    times;
    initial_condition=:m,
    reset=delta_reset_protocol,
)

@assert k in ame.k_values "Requested k is not present in AME degree classes"
k_idx_ame = findfirst(==(k), ame.k_values)

s4_ame = ame.s_values[k_idx_ame]  # (ntimes, k+1)
i4_ame = ame.i_values[k_idx_ame]  # (ntimes, k+1)

sum_s_ame = vec(sum(s4_ame, dims=2))
sum_i_ame = vec(sum(i4_ame, dims=2))
sum_si_ame = sum_s_ame + sum_i_ame
m_from_si_ame = vec(sum(i4_ame .- s4_ame, dims=2))

# Active-link density for a k-regular graph: rho(t) = (1/k) * sum_m m * [s_{k,m}(t) + i_{k,m}(t)]
m_vals = collect(0:k)
rho_active_sim = vec((s4 + i4) * m_vals) ./ k
rho_active_ame = vec((s4_ame + i4_ame) * m_vals) ./ k

@assert length(result.times) == length(ame.times)

println("Comparison diagnostics (max abs diff over time):")
println("  using shared user-facing r=$r and explicit delta reset")
for m in selected_s_m
    ds = maximum(abs.(s4[:, m + 1] .- s4_ame[:, m + 1]))
    @printf("  m=%d  max|Δs|=%.3e\n", m, ds)
end
@printf("  max|Δ(sum_s)|=%.3e\n", maximum(abs.(sum_s .- sum_s_ame)))
@printf("  max|Δ(sum_i)|=%.3e\n", maximum(abs.(sum_i .- sum_i_ame)))
@printf("  max|Δ(m)|    =%.3e\n", maximum(abs.(m_from_si .- m_from_si_ame)))
@printf("  max|Δ(ρ_active)|=%.3e\n", maximum(abs.(rho_active_sim .- rho_active_ame)))

# Only selected s_{k,m}(t): simulation (solid) vs AME (dashed)
p_s_cmp = plot(size=(950, 450), legend=:right, xlabel="t", ylabel="fraction",
    title="RRG k=4 (delta reset): selected s_{4,m}(t) — simulation vs AME")
for m in selected_s_m
    plot!(p_s_cmp, result.times, s4[:, m + 1], lw=2.5, label="sim s_{$k,$m}")
    plot!(p_s_cmp, result.times, s4_ame[:, m + 1], lw=2.5, ls=:dash, label="AME s_{$k,$m}")
end
display(p_s_cmp)

# Aggregate comparison
p_agg_cmp = plot(size=(950, 450), legend=:right, xlabel="t", ylabel="value",
    title="RRG k=4 (delta reset): aggregates — simulation vs AME")
plot!(p_agg_cmp, result.times, sum_s, lw=2, label="sim sum_m s")
plot!(p_agg_cmp, result.times, sum_s_ame, lw=2, ls=:dash, label="AME sum_m s")
plot!(p_agg_cmp, result.times, sum_i, lw=2, label="sim sum_m i")
plot!(p_agg_cmp, result.times, sum_i_ame, lw=2, ls=:dash, label="AME sum_m i")
plot!(p_agg_cmp, result.times, m_from_si, lw=2, color=:black, label="sim m(t)")
plot!(p_agg_cmp, result.times, m_from_si_ame, lw=2, ls=:dash, color=:black, label="AME m(t)")
plot!(p_agg_cmp, result.times, sum_si, lw=2, ls=:dot, color=:gray, label="sim sum_m(s+i)")
plot!(p_agg_cmp, result.times, sum_si_ame, lw=2, ls=:dashdot, color=:gray, label="AME sum_m(s+i)")
hline!(p_agg_cmp, [m0], ls=:dashdot, lw=1.5, color=:red, label="m0")
display(p_agg_cmp)

# Active-link density comparison
p_rho_cmp = plot(
    size=(950, 450),
    legend=:right,
    xlabel="t",
    ylabel="active-link density ρ(t)",
    title="RRG k=4 (delta reset): active-link density — simulation vs AME",
)
plot!(p_rho_cmp, result.times, rho_active_sim, lw=2.6, label="sim ρ_active")
plot!(p_rho_cmp, result.times, rho_active_ame, lw=2.6, ls=:dash, label="AME ρ_active")
display(p_rho_cmp)

In [ ]:
# Diagnostic A: AME with and without k*t.
ame_kt = VoterResetting.solve_ame_evolution(
    G,
    m0,
    r,
    k .* times;
    initial_condition=:m,
    reset=delta_reset_protocol,
)

k_idx_kt = findfirst(==(k), ame_kt.k_values)
s4_kt = ame_kt.s_values[k_idx_kt]
i4_kt = ame_kt.i_values[k_idx_kt]

sum_s_kt = vec(sum(s4_kt, dims=2))
sum_i_kt = vec(sum(i4_kt, dims=2))
sum_si_kt = sum_s_kt + sum_i_kt
m_from_si_kt = vec(sum(i4_kt .- s4_kt, dims=2))

println("Diagnostic A:")
for m in selected_s_m
    d_nonkt = maximum(abs.(s4[:, m + 1] .- s4_ame[:, m + 1]))
    d_kt = maximum(abs.(s4[:, m + 1] .- s4_kt[:, m + 1]))
    @printf("  m=%d  non-k*t=%.3e  k*t=%.3e\n", m, d_nonkt, d_kt)
end

p_diagA = plot(
    size=(950, 450),
    legend=:right,
    xlabel="t",
    ylabel="fraction",
    title="Diagnostic A: simulation vs AME (non-k*t and k*t)",
)
for m in selected_s_m
    plot!(p_diagA, result.times, s4[:, m + 1], lw=2.8, label="sim s_{$k,$m}")
    plot!(p_diagA, result.times, s4_ame[:, m + 1], lw=2.2, ls=:dash, label="AME s_{$k,$m}")
    plot!(p_diagA, result.times, s4_kt[:, m + 1], lw=2.0, ls=:dashdot, label="AME(k*t) s_{$k,$m}")
end

display(p_diagA)

p_diagA_agg = plot(
    size=(950, 450),
    legend=:right,
    xlabel="t",
    ylabel="value",
    title="Diagnostic A aggregates: AME non-k*t vs k*t",
)
plot!(p_diagA_agg, result.times, sum_s, lw=2, label="sim sum_m s")
plot!(p_diagA_agg, result.times, sum_s_ame, lw=2, ls=:dash, label="AME sum_m s")
plot!(p_diagA_agg, result.times, sum_s_kt, lw=2, ls=:dashdot, label="AME(k*t) sum_m s")
plot!(p_diagA_agg, result.times, sum_i, lw=2, label="sim sum_m i")
plot!(p_diagA_agg, result.times, sum_i_ame, lw=2, ls=:dash, label="AME sum_m i")
plot!(p_diagA_agg, result.times, sum_i_kt, lw=2, ls=:dashdot, label="AME(k*t) sum_m i")
plot!(p_diagA_agg, result.times, m_from_si, lw=2, color=:black, label="sim m(t)")
plot!(p_diagA_agg, result.times, m_from_si_ame, lw=2, ls=:dash, color=:black, label="AME m(t)")
plot!(p_diagA_agg, result.times, m_from_si_kt, lw=2, ls=:dashdot, color=:black, label="AME(k*t) m(t)")
plot!(p_diagA_agg, result.times, sum_si, lw=2, ls=:dot, color=:gray, label="sim sum_m(s+i)")
plot!(p_diagA_agg, result.times, sum_si_ame, lw=2, ls=:dash, color=:gray, label="AME sum_m(s+i)")
plot!(p_diagA_agg, result.times, sum_si_kt, lw=2, ls=:dashdot, color=:gray, label="AME(k*t) sum_m(s+i)")
hline!(p_diagA_agg, [m0], ls=:dashdot, lw=1.5, color=:red, label="m0")
display(p_diagA_agg)

In [ ]:
# Diagnostic B: early-time slope check (edge-update AME only).
function local_s_indicator(state, neighbors_cache, node, m_target)
    m_plus = count(nb -> state[nb] == Int8(1), neighbors_cache[node])
    return (state[node] == Int8(-1) && m_plus == m_target) ? (1.0 / length(state)) : 0.0
end

function delta_selected_s_for_flip!(state, neighbors_cache, node, selected_m)
    affected_nodes = [node; neighbors_cache[node]]
    before = Dict(
        m => sum(local_s_indicator(state, neighbors_cache, v, m) for v in affected_nodes)
        for m in selected_m
    )

    old_state = state[node]
    state[node] = Int8(-old_state)

    after = Dict(
        m => sum(local_s_indicator(state, neighbors_cache, v, m) for v in affected_nodes)
        for m in selected_m
    )

    state[node] = old_state
    return Dict(m => after[m] - before[m] for m in selected_m)
end

function exact_initial_edge_generator_slopes(G, m0, selected_m; nsamples=1200, seed=12345)
    Random.seed!(seed)
    neighbors_cache = [collect(neighbors(G, v)) for v in 1:nv(G)]
    slope_acc = Dict(m => 0.0 for m in selected_m)

    for _ in 1:nsamples
        state = VoterResetting.random_spin_state(nv(G), m0)
        for edge in edges(G)
            u = Int(src(edge))
            v = Int(dst(edge))
            state[u] == state[v] && continue

            du = delta_selected_s_for_flip!(state, neighbors_cache, u, selected_m)
            dv = delta_selected_s_for_flip!(state, neighbors_cache, v, selected_m)
            for m in selected_m
                slope_acc[m] += du[m] + dv[m]
            end
        end
    end

    return Dict(m => slope_acc[m] / nsamples for m in selected_m)
end

edge_generator_slopes = exact_initial_edge_generator_slopes(G, m0, selected_s_m; nsamples=1500, seed=seed + 99)
dt = times[2] - times[1]

println("Diagnostic B: early-time slopes (edge-update AME only)")
for m in selected_s_m
    sim_fd = (s4[2, m + 1] - s4[1, m + 1]) / dt
    ame_edge_fd = (s4_ame[2, m + 1] - s4_ame[1, m + 1]) / dt
    ame_kt_fd = (s4_kt[2, m + 1] - s4_kt[1, m + 1]) / dt
    exact_edge_rhs = edge_generator_slopes[m]

    @printf("  m=%d\n", m)
    @printf("    exact edge generator : %.6e\n", exact_edge_rhs)
    @printf("    sim finite diff      : %.6e\n", sim_fd)
    @printf("    AME fd edge          : %.6e\n", ame_edge_fd)
    @printf("    AME fd edge(k*t)     : %.6e\n", ame_kt_fd)
end

p_diagB = plot(
    size=(950, 450),
    legend=:right,
    xlabel="t",
    ylabel="fraction",
    title="Diagnostic B: early-time window (edge AME non-k*t vs k*t)",
    xlim=(0.0, min(1.5, maximum(times))),
)
for m in selected_s_m
    plot!(p_diagB, result.times, s4[:, m + 1], lw=2.8, label="sim s_{$k,$m}")
    plot!(p_diagB, result.times, s4_ame[:, m + 1], lw=2.2, ls=:dash, label="AME edge s_{$k,$m}")
    plot!(p_diagB, result.times, s4_kt[:, m + 1], lw=2.0, ls=:dashdot, label="AME edge(k*t) s_{$k,$m}")
end

display(p_diagB)